In [4]:
#!/usr/bin/env python3

from pathlib import Path
import random
import numpy as np
import librosa
import soundfile as sf
import shutil

# -------------------------
# Settings
# -------------------------
PREPARED_WORDS_DIR = Path("prepared_words")
BACKGROUND_NOISE_DIR = Path("background_noise")
OUTPUT_DIR = Path("keywords_augmented")

TARGET_LABELS = ["får", "ged", "hest", "laks", "ulv"]
NOISE_LABELS = ["nygaard", "tmlKlasse"]

SAMPLE_RATE = 16000
SAMPLE_TIME = 1.0
WORD_VOL = 1.0
BG_VOL = 0.1
BIT_DEPTH = "PCM_16"

# -------------------------
# Helpers
# -------------------------
def load_audio_mono(path: Path, sample_rate: int, sample_time: float):
    """Load wav, resample to sample_rate, force mono, pad/truncate to sample_time."""
    y, _ = librosa.load(str(path), sr=sample_rate, mono=True)
    target_len = int(sample_rate * sample_time)

    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]

    return y

def get_random_bg_snippet(bg_path: Path, sample_rate: int, sample_time: float):
    """Load background wav and return a random snippet of sample_time seconds."""
    bg, _ = librosa.load(str(bg_path), sr=sample_rate, mono=True)
    target_len = int(sample_rate * sample_time)

    if len(bg) < target_len:
        raise ValueError(f"Background file is shorter than {sample_time}s: {bg_path}")

    max_start = len(bg) - target_len
    start = random.randint(0, max_start)
    end = start + target_len
    return bg[start:end]

def mix_audio(word_waveform, bg_waveform, word_vol=1.0, bg_vol=0.1):
    mixed = 0.5 * word_vol * word_waveform + 0.5 * bg_vol * bg_waveform
    mixed = np.clip(mixed, -1.0, 1.0)
    return mixed

def replace_noise_label(filename: str, new_noise_label: str):
    """
    Replace '_stille_' with '_nygaard_' or '_tmlKlasse_'.
    If '_stille_' is not found, append the noise label before file extension.
    """
    if "_stille_" in filename:
        return filename.replace("_stille_", f"_{new_noise_label}_", 1)

    stem = Path(filename).stem
    suffix = Path(filename).suffix
    return f"{stem}_{new_noise_label}{suffix}"

def collect_wavs(folder: Path):
    return sorted([p for p in folder.glob("*.wav") if p.is_file()])

# -------------------------
# Main
# -------------------------
def main():
    random.seed()

    if OUTPUT_DIR.exists():
        print(f"Deleting existing output folder: {OUTPUT_DIR}")
        shutil.rmtree(OUTPUT_DIR)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    total_original = 0
    total_augmented = 0

    # Collect background files by noise type
    bg_files = {}
    for noise_label in NOISE_LABELS:
        noise_dir = BACKGROUND_NOISE_DIR / noise_label
        files = collect_wavs(noise_dir)
        if not files:
            raise FileNotFoundError(f"No WAV files found in {noise_dir}")
        bg_files[noise_label] = files

    for label in TARGET_LABELS:
        in_dir = PREPARED_WORDS_DIR / label
        out_dir = OUTPUT_DIR / label
        out_dir.mkdir(parents=True, exist_ok=True)

        word_files = collect_wavs(in_dir)
        if not word_files:
            print(f"Skipping empty label folder: {in_dir}")
            continue

        print(f"\nProcessing label: {label} ({len(word_files)} files)")

        for word_path in word_files:
            # 1) Copy original "stille" version
            shutil.copy2(word_path, out_dir / word_path.name)
            total_original += 1

            # Load original keyword once
            word_waveform = load_audio_mono(word_path, SAMPLE_RATE, SAMPLE_TIME)

            # 2) Create one augmented version per noise label
            for noise_label in NOISE_LABELS:
                bg_path = random.choice(bg_files[noise_label])
                bg_waveform = get_random_bg_snippet(bg_path, SAMPLE_RATE, SAMPLE_TIME)

                mixed = mix_audio(
                    word_waveform,
                    bg_waveform,
                    word_vol=WORD_VOL,
                    bg_vol=BG_VOL
                )

                out_name = replace_noise_label(word_path.name, noise_label)
                out_path = out_dir / out_name

                sf.write(str(out_path), mixed, SAMPLE_RATE, subtype=BIT_DEPTH)
                total_augmented += 1

    print("\nDone.")
    print(f"Original keyword files copied: {total_original}")
    print(f"Augmented files created:       {total_augmented}")
    print(f"Total files in output:         {total_original + total_augmented}")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'librosa'